In [10]:
from typing_extensions import TypedDict
from typing import List
from langgraph.graph import StateGraph, START, END
from langchain.chat_models import init_chat_model
from pydantic import BaseModel

llm = init_chat_model("openai:gpt-4o-mini")


In [11]:

class State(TypedDict) :
    
    dish : str
    ingredients : list[dict]               # 재료
    recipe_steps : list[dict]              # 요리 순서
    plating_instructions : str      # 플레이팅 지침

class Ingredient(BaseModel) :
    name : str
    quantity : str
    unit : str

# 구조화된 출력을 위한 구조체 [재료 목록]
class IngredientList(BaseModel) :
    ingredients : List[Ingredient]

# 조리법 각 단계
class Recipe_Step(BaseModel) :
    step : int              # 몇번째 순서인지
    instruction : str       # 지침

# 조리법
class Recipe(BaseModel):
    steps : List[Recipe_Step] # 각단계의 모음음


In [12]:
def node_prepare_ingredients(state : State) :
    # 재료를 준비합니다.

    llm_ingredientsList = llm.with_structured_output(IngredientList)

    received_ingredientList = llm_ingredientsList.invoke(
        f" 음식 {state['dish']} 을 만들기 위한 n개의 재료를 준비해봐"
    )

    return {
        "ingredients" : received_ingredientList.ingredients
    }

def node_write_recipe(state:State) :

    recipe_llm = llm.with_structured_output(Recipe)

    writed_recipe = recipe_llm.invoke(
        f"제공하는 재료들({state['ingredients']}) 를 사용하여 요리 '{state['dish']}' 를 단계별로 조리하는 방법을 작성하세요."
    )

    return {
        "recipe_steps" : writed_recipe.steps
    }

def node_describe_plating(state : State):
    
    llm_plating = llm.invoke(
        f"이 레시피 ({state['recipe_steps']}) 을 바탕으로 이 요리 '{state["dish"]}' 를 아름답게 플레이팅 하는 방법을 설명하세요"
    )

    return {
        "plating_instructions" : llm_plating.content
    }


In [13]:
graph_builder = StateGraph(State)

graph_builder.add_node("node_prepare_ingredients",node_prepare_ingredients)
graph_builder.add_node("node_write_recipe",node_write_recipe)
graph_builder.add_node("node_describe_plating",node_describe_plating)

graph_builder.add_edge(START, "node_prepare_ingredients")
graph_builder.add_edge("node_prepare_ingredients","node_write_recipe")
graph_builder.add_edge("node_write_recipe","node_describe_plating")
graph_builder.add_edge("node_describe_plating",END)


graph = graph_builder.compile()

In [14]:
graph.invoke({"dish":"닭볶음탕"})

c:\HeukKell_Asset\ProjectFolder\Temporal\Python\GitHub\Langgraph_Workflow\.venv\Lib\site-packages\pydantic\main.py:464: UserWarning: Pydantic serializer warnings:
  PydanticSerializationUnexpectedValue(Expected `none` - serialized value may not be as expected [field_name='parsed', input_value=IngredientList(ingredient...ntity='4', unit='컵')]), input_type=IngredientList])
  return self.__pydantic_serializer__.to_python(
c:\HeukKell_Asset\ProjectFolder\Temporal\Python\GitHub\Langgraph_Workflow\.venv\Lib\site-packages\pydantic\main.py:464: UserWarning: Pydantic serializer warnings:
  PydanticSerializationUnexpectedValue(Expected `none` - serialized value may not be as expected [field_name='parsed', input_value=Recipe(steps=[Recipe_Step...장식해 주세요.')]), input_type=Recipe])
  return self.__pydantic_serializer__.to_python(


{'dish': '닭볶음탕',
 'ingredients': [Ingredient(name='닭고기 (통닭 또는 닭날개)', quantity='1', unit='kg'),
  Ingredient(name='감자', quantity='2', unit='개'),
  Ingredient(name='당근', quantity='1', unit='개'),
  Ingredient(name='양파', quantity='1', unit='개'),
  Ingredient(name='대파', quantity='1', unit='대'),
  Ingredient(name='청양고추', quantity='2', unit='개'),
  Ingredient(name='간장', quantity='5', unit='큰술'),
  Ingredient(name='고춧가루', quantity='2', unit='큰술'),
  Ingredient(name='설탕', quantity='1', unit='큰술'),
  Ingredient(name='마늘', quantity='5', unit='쪽'),
  Ingredient(name='생강', quantity='1', unit='쪽'),
  Ingredient(name='물', quantity='4', unit='컵')],
 'recipe_steps': [Recipe_Step(step=1, instruction='닭고기를 씻고, 먹기 좋은 크기로 잘라 주세요.'),
  Recipe_Step(step=2, instruction='감자와 당근은 껍질을 벗기고 큼직하게 깍둑썰기 해 주세요.'),
  Recipe_Step(step=3, instruction='양파는 굵게 채썰고, 대파는 어슷하게 썰어 주세요.'),
  Recipe_Step(step=4, instruction='청양고추는 어슷하게 썰어 주세요.'),
  Recipe_Step(step=5, instruction='마늘과 생강은 다져서 준비해 주세요.'),
  Recipe_Step(step=6, in